# 04 — Análise do Grafo e Detecção de Comunidades

Calculamos métricas de centralidade e detectamos comunidades com o algoritmo de Louvain.
Execute os notebooks `01_coleta` e `03_grafo` antes deste.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import networkx as nx
from pathlib import Path

from src.fetch import carregar_raw
from src.graph import construir_grafo_coautoria, filtrar_grafo, maior_componente
from src.metrics import calcular_centralidades, detectar_comunidades, resumo_rede

FIGURES = Path('../reports/figures')
PROCESSED = Path('../data/processed')
PROCESSED.mkdir(parents=True, exist_ok=True)

works = carregar_raw('works_ufms_cs')
G = maior_componente(filtrar_grafo(construir_grafo_coautoria(works), min_degree=2))
print(resumo_rede(G))

## 1. Métricas de Centralidade

In [ ]:
df = calcular_centralidades(G)
df.to_csv(PROCESSED / 'centralidades.csv', index=False)
print(f'Exportado: {PROCESSED}/centralidades.csv')
df.head(15)

## 2. Distribuição de Centralidade

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col, label in zip(axes,
    ['degree', 'betweenness', 'eigenvector'],
    ['Degree', 'Betweenness', 'Eigenvector']
):
    df[col].plot(kind='hist', bins=20, color='#01696f', ax=ax)
    ax.set_title(f'Distribuição — {label}', fontsize=12)
    ax.set_xlabel(label)

plt.tight_layout()
plt.savefig(FIGURES / 'distribuicao_centralidades.png', dpi=150)
plt.show()

## 3. Detecção de Comunidades (Louvain)

In [ ]:
comunidades = detectar_comunidades(G)
nx.set_node_attributes(G, comunidades, 'community')

n_com = len(set(comunidades.values()))
print(f'Comunidades detectadas: {n_com}')

# Distribuição de tamanho das comunidades
tamanhos = pd.Series(comunidades.values()).value_counts().sort_index()
print(tamanhos.to_string())

In [ ]:
pos = nx.spring_layout(G, seed=42, k=0.6)
cores = [comunidades[n] for n in G.nodes()]

plt.figure(figsize=(14, 10))
nx.draw_networkx(
    G, pos=pos,
    node_color=cores, cmap=cm.tab20,
    node_size=[G.degree(n) * 20 + 50 for n in G.nodes()],
    edge_color='#dddddd', width=0.5,
    font_size=6, with_labels=True,
)
plt.title(f'Grafo de Co-autoria — {n_com} comunidades (Louvain)', fontsize=14)
plt.axis('off')
plt.tight_layout()
plt.savefig(FIGURES / 'comunidades.png', dpi=150)
plt.show()

## 4. Exportar Grafo como CSV de Arestas

In [ ]:
edges_df = pd.DataFrame([
    {
        'source': u,
        'target': v,
        'weight': d.get('weight', 1),
        'community_source': comunidades.get(u, -1),
        'community_target': comunidades.get(v, -1),
    }
    for u, v, d in G.edges(data=True)
])

edges_df.to_csv(PROCESSED / 'arestas_coautoria.csv', index=False)
print(f'{len(edges_df)} arestas exportadas')
edges_df.head()

## 5. Top Brokers (Alta Betweenness)

In [ ]:
print('Pesquisadores com maior betweenness (pontes entre grupos):\n')
print(df[['node', 'degree_raw', 'betweenness', 'eigenvector']].head(10).to_string(index=False))